### **PDF Splitting for Larger PDF**

In [6]:
import os, requests, math, io 
from urllib.parse import urlparse 
from PyPDF2 import PdfReader, PdfWriter

In [7]:
def get_pdf_reader(input_source):
    """ 
        Gets a PdfReader object from a URL.
        Args:
            input_source(str) : The URL of the PDF 
        Returns:
            tuple : (PdfReader object , base filename , total pages)
    """

    base_filename = "output"
    response = requests.get(input_source , stream = True , timeout=30)
    response.raise_for_status() # Raise exception if any error 

    # Get the file name from the URL path
    parsed_url = urlparse(input_source)
    path_part = os.path.basename(parsed_url.path)

    # Basic check for the file name
    if path_part and '.' in path_part:
        base_filename = os.path.splitext(path_part)[0]

    # Read the content into memory 
    pdf_content = io.BytesIO(response.content)
    reader = PdfReader(pdf_content)
    total_pages = len(reader.pages)
    print(f"Successfully downloaded and opened PDF from URL. Total pages : {total_pages}")
    return reader, base_filename , total_pages

In [8]:
def split_pdf(input_source , output_dir , pages_per_chunk):
    """ 
        Splits a PDF file into multiple smaller pdf file.
        Args:
            input_source(str) : The path to the local PDF file or the URL of the PDF 
            output_dir(str) : The directory where the smaller PDF chunks will be saved.
            pages_per_chunk(int) : The maximum number of pages each chunk should have.
    """
    
    # Getting the variable from the get pdf reader function
    reader , base_filename , total_pages = get_pdf_reader(input_source)

    if reader is None:
        print("Failed to get PDF reader. Abborting the split...")
        return 
    
    try:
        # Create the output directory if it doesnot exist
        os.makedirs(output_dir , exist_ok=True)
        print(f"Output directory {output_dir} ensured")

        # Calculate the number of chunks
        num_chunks = math.ceil(total_pages / pages_per_chunk)
        print(f"Splitting into {num_chunks} chunks of max {pages_per_chunk} page each.")

        # process each chunk
        for i in range(num_chunks):
            writer = PdfWriter()
            start_page = i * pages_per_chunk
            # Ensure end_page doent exceed total_pages 
            end_page = min(start_page + pages_per_chunk , total_pages)

            print(f"Processing chunk {i+1} / {num_chunks} (pages {start_page + 1} - {end_page})...")

            # Add pages to the new PDF chunk
            for page_num in range(start_page , end_page):
                writer.add_page(reader.pages[page_num])

            # Construct the output file name using the determined base filename
            output_filename = os.path.join(output_dir , f"{base_filename}_chunk+{i+1}.pdf")

            # Write the chunk to a new PDF file 
            with open(output_filename , "wb") as outfile:
                writer.write(outfile)
            print(f"Chunk {i+1} saved as {output_filename}")

        print("\nPDF splitting completed successfully!")

    except Exception as e:
        print(f"An error occured during the splitting process : {e}")

In [9]:
pdf_link = "https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf"
output_folder = "pdf_chunks"
pages_per_split = 50
split_pdf(pdf_link , output_folder , pages_per_split)

Successfully downloaded and opened PDF from URL. Total pages : 352
Output directory pdf_chunks ensured
Splitting into 8 chunks of max 50 page each.
Processing chunk 1 / 8 (pages 1 - 50)...
Chunk 1 saved as pdf_chunks\SuttonBartoIPRLBook2ndEd_chunk+1.pdf
Processing chunk 2 / 8 (pages 51 - 100)...
Chunk 2 saved as pdf_chunks\SuttonBartoIPRLBook2ndEd_chunk+2.pdf
Processing chunk 3 / 8 (pages 101 - 150)...
Chunk 3 saved as pdf_chunks\SuttonBartoIPRLBook2ndEd_chunk+3.pdf
Processing chunk 4 / 8 (pages 151 - 200)...
Chunk 4 saved as pdf_chunks\SuttonBartoIPRLBook2ndEd_chunk+4.pdf
Processing chunk 5 / 8 (pages 201 - 250)...
Chunk 5 saved as pdf_chunks\SuttonBartoIPRLBook2ndEd_chunk+5.pdf
Processing chunk 6 / 8 (pages 251 - 300)...
Chunk 6 saved as pdf_chunks\SuttonBartoIPRLBook2ndEd_chunk+6.pdf
Processing chunk 7 / 8 (pages 301 - 350)...
Chunk 7 saved as pdf_chunks\SuttonBartoIPRLBook2ndEd_chunk+7.pdf
Processing chunk 8 / 8 (pages 351 - 352)...
Chunk 8 saved as pdf_chunks\SuttonBartoIPRLBook2n